In [1]:
import os
import json
import random
from typing import List, Dict, Any

In [2]:
def format_nl_premises(premises_nl: List[str]) -> str:
    """Format premises as a numbered list string."""
    return "\n".join(f"{i + 1}. {premise}" for i, premise in enumerate(premises_nl))

def format_fol_premises(premises_fol: List[str]) -> str:
    """Format FOL premises as a numbered list string."""
    return "\n".join(f"{i + 1}. {premise}" for i, premise in enumerate(premises_fol))

def build_unified_sample(record: Dict[str, Any]) -> Dict[str, Any]:
    """
    Build conversational message format for a single-pass joint-output model.
    It combines premises_fol, explanation inside think, Z3 code, and final answer.
    """
    premises_nl = record.get("premises-NL", [])
    premises_fol = record.get("premises-FOL", [])
    questions = record.get("questions", [])
    answers = record.get("answers", [])
    explanations = record.get("explanation", [])
    idx_list = record.get("idx", [])
    z3_codes = record.get("z3_code", [])

    # 1. System Prompt guiding the model to reason inside <think> and output structured XML
    system_prompt = (
        "You are a logical reasoning expert. Given a set of premises in natural language and a list of questions:\n"
        "1. You must first think step-by-step inside the <think>...</think> tags. In this thinking block, translate the premises into First-Order Logic (FOL), analyze the logical relationships, draft and debug the Python Z3 code, and determine the correct answers.\n"
        "2. After closing the </think> tag, output the finalized structured results using XML tags:\n"
        "   - Use <premises_fol>...</premises_fol> to list the FOL translations of all premises.\n"
        "   - Use <question_outputs>...</question_outputs> containing a <question id=\"i\"> block for each question.\n"
        "   - Within each question block, provide the pedagogical <explanation> for the user, the list of <premises_used>, the executable <z3_code>, and the final <answer>.\n\n"
        "Important Guidelines:\n"
        "- In the <explanation> tag, always explain step-by-step how the answer is derived and explicitly cite the premise numbers you used (e.g., 'From premise 3, we know that...'). Do not refer to premises without their index numbers.\n"
        "- Ensure that you use consistent variable names in the Z3 code corresponding to the entities and relations in the premises."
    )

    # 2. User Prompt containing premises and list of questions
    formatted_premises_nl = format_nl_premises(premises_nl)
    questions_str = "\n".join(f"Question {i + 1}: {q}" for i, q in enumerate(questions))
    user_content = f"Premises (Natural Language):\n{formatted_premises_nl}\n\nQuestions:\n{questions_str}"

    # 3. Assistant Target content
    # Combine original explanations to form the CoT thinking block
    cot_thinking = "\n".join(
        f"For Question {i + 1}: {exp.strip()}" 
        for i, exp in enumerate(explanations)
    )
    
    formatted_premises_fol = format_fol_premises(premises_fol)
    
    q_outputs = []
    for i in range(len(questions)):
        q_outputs.append(
            f'  <question id="{i + 1}">\n'
            f'    <explanation>\n      {explanations[i].strip()}\n    </explanation>\n'
            f'    <premises_used>{json.dumps(idx_list[i])}</premises_used>\n'
            f'    <z3_code>\n{z3_codes[i].strip()}\n    </z3_code>\n'
            f'    <answer>{answers[i].strip()}</answer>\n'
            f'  </question>'
        )
    q_outputs_str = "\n".join(q_outputs)

    assistant_content = (
        f"<think>\n{cot_thinking}\n</think>\n\n"
        f"<premises_fol>\n{formatted_premises_fol}\n</premises_fol>\n\n"
        f"<question_outputs>\n{q_outputs_str}\n</question_outputs>"
    )

    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content}
        ]
    }

def prepare_data(
    input_file: str,
    output_dir: str,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42
):
    """Load, process, shuffle and split dataset at the record level."""
    print(f"Loading data from {input_file}...")
    with open(input_file, "r", encoding="utf-8") as f:
        records = json.load(f)

    # Process all records into the unified template
    samples = [build_unified_sample(rec) for rec in records]
    print(f"Processed {len(samples)} records successfully.")

    # Split records
    random.seed(seed)
    indices = list(range(len(samples)))
    random.shuffle(indices)

    total = len(samples)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]

    print(f"Split: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")

    # Ensure output dir exists
    os.makedirs(output_dir, exist_ok=True)

    splits = {
        "logic_train.jsonl": train_idx,
        "logic_val.jsonl": val_idx,
        "logic_test.jsonl": test_idx
    }

    for filename, idx_list in splits.items():
        filepath = os.path.join(output_dir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            for idx in idx_list:
                f.write(json.dumps(samples[idx], ensure_ascii=False) + "\n")
        print(f"Saved {len(idx_list)} samples to {filepath}")


In [3]:
workspace_dir = r"d:\Education\exact_2026"
input_json = os.path.join(workspace_dir, "data", "processed", "Logic_SFT.json")
output_directory = os.path.join(workspace_dir, "data", "processed")

prepare_data(
    input_file=input_json,
    output_dir=output_directory,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42
)

Loading data from d:\Education\exact_2026\data\processed\Logic_SFT.json...
Processed 864 records successfully.
Split: Train=691, Val=86, Test=87
Saved 691 samples to d:\Education\exact_2026\data\processed\logic_train.jsonl
Saved 86 samples to d:\Education\exact_2026\data\processed\logic_val.jsonl
Saved 87 samples to d:\Education\exact_2026\data\processed\logic_test.jsonl
